In [2]:
import os
import sys
from pathlib import Path


def get_current_path() -> Path:
    """
    Returns the Path of the current .py script or .ipynb notebook.
    """

    # ── Jupyter / IPython environment ────────────────────────────────────────
    try:
        from IPython import get_ipython
        ipy = get_ipython()

        if ipy is not None:
            # 1. VS Code notebooks
            vscode_path = os.environ.get("VSCODE_NOTEBOOK_PATH")
            if vscode_path:
                return Path(vscode_path).resolve()

            # 2. Jupyter Lab / Notebook — query the running server's API
            try:
                import json
                import urllib.request
                from jupyter_server import serverapp
                from jupyter_server.utils import url_path_join

                kernel_id = Path(
                    ipy.config["IPKernelApp"]["connection_file"]
                ).stem.replace("kernel-", "")

                for server in serverapp.list_running_servers():
                    url = url_path_join(server["url"], "api/sessions")
                    req = urllib.request.Request(
                        url,
                        headers={"Authorization": f"token {server.get('token', '')}"}
                    )
                    sessions = json.loads(urllib.request.urlopen(req).read())
                    for sess in sessions:
                        if kernel_id in sess.get("kernel", {}).get("id", ""):
                            return Path(server["root_dir"]) / sess["notebook"]["path"]
            except Exception:
                pass

    except ImportError:
        pass

    # ── Plain .py script ─────────────────────────────────────────────────────
    frame = sys._getframe(1)
    script_path = frame.f_globals.get("__file__")
    if script_path:
        return Path(script_path).resolve()

    raise RuntimeError("Could not determine the current file path.")

In [4]:
get_current_path()

RuntimeError: Could not determine the current file path.